In [20]:
# SPUR DuckDB extension setup cell v1
# requirements: {"active":[{"packageName":"duckrouting","source":"community"},{"packageName":"shellfs","source":"community"}],"pending":[]}
# Managed by Jute. Change packages through the extension sidebar.
# SPUR DuckDB bootstrap
import duckdb
import sys
_SPUR_DUCKDB_EXTENSION_PATH = "/Users/kevintruong/.spur/extensions/spur_rest.duckdb_extension"
# Shared by notebook setup and classic SQL cells. Preserve the generic path for
# older bundles; new bundles carry exact native ABI artifacts per engine.
from pathlib import Path as _SpurExtensionPath

_spur_extension = _SpurExtensionPath(_SPUR_DUCKDB_EXTENSION_PATH)
_spur_versioned_extension = (
    _spur_extension.parent
    / ("duckdb-v" + duckdb.__version__)
    / _spur_extension.name
)
if _spur_versioned_extension.is_file():
    _SPUR_DUCKDB_EXTENSION_PATH = str(_spur_versioned_extension)

if (
    sys.platform in ("linux", "darwin")
    and duckdb.__version__ in ("1.4.4", "1.5.5")
    and "_SPUR_DUCKDB_HOST_SYMBOLS" not in globals()
):
    import ctypes as _spur_ctypes
    import _duckdb as _spur_host_duckdb

    # Promote the already-loaded Python engine's symbols. This opens the same
    # module handle; it neither imports another engine nor replaces connections.
    _SPUR_DUCKDB_HOST_SYMBOLS = _spur_ctypes.CDLL(
        _spur_host_duckdb.__file__, mode=_spur_ctypes.RTLD_GLOBAL
    )

_SPUR_DUCKDB_EXTENSION_SQL = _SPUR_DUCKDB_EXTENSION_PATH.replace("'", "''")

if "_SPUR_DUCKDB_CONNECTION" not in globals():
    _SPUR_DUCKDB_CONNECTION = duckdb.connect(
        database=":memory:",
        config={"allow_unsigned_extensions": "true",
                 "autoload_known_extensions": "false",
                 "autoinstall_known_extensions": "false"},
    )

duckdb.set_default_connection(_SPUR_DUCKDB_CONNECTION)
# Embedded helper source contains no token, credential path or capability lease.
# The approved local launcher passes a private inherited socket descriptor, not
# executable source or a notebook/widget message. Same-kernel code is trusted.
def _spur_managed_connection(db):
    """Only the host's selected connection is subject to managed policy."""
    import types
    connection = db.default_connection() if isinstance(db, types.ModuleType) else db
    if connection is not globals().get("_SPUR_DUCKDB_CONNECTION"):
        return None
    try:
        connection.sql("SET autoload_known_extensions = false")
        connection.sql("SET autoinstall_known_extensions = false")
    except Exception:
        return None  # existing locked/custom connections have an explicit boundary
    return connection


# Required SpurLab baseline: JSON/parquet are statically linked in the managed
# wheel; httpfs supplies remote files and REST gateway dependencies; spur_rest is
# the separately packaged, host-verified gateway (never a repository download).
# Remote datasource attach code explicitly installs/loads its provider adapter.
# Spatial and Jev are optional. No browsing/query path calls operate("install").
# Fixed metadata query only. Identifiers and descriptions remain inert JSON; no
# catalog value is interpolated as SQL. The limits bound the shell reply too.
def _spur_function_catalog(db):
    import json
    limit = 10000
    rows = db.sql("SELECT database_name, schema_name, function_name, function_type, "
                  "parameters, parameter_types, varargs, return_type, description, examples, function_oid "
                  "FROM duckdb_functions() ORDER BY database_name, schema_name, function_name, "
                  "function_type, parameter_types, varargs LIMIT 10001").fetchall()
    functions, size = [], 0
    complete = len(rows) <= limit
    for row in rows[:limit]:
        value = dict(zip(("database_name", "schema_name", "name", "function_type", "parameters",
                          "parameter_types", "varargs", "return_type", "description", "examples", "function_oid"), row))
        value["ownership"] = "unknown"
        encoded = json.dumps(value, ensure_ascii=True)
        # Oversized metadata is omitted, with incomplete coverage made explicit.
        if len(encoded) > 65536 or size + len(encoded) > 4 * 1024 * 1024:
            complete = False
            continue
        functions.append(value)
        size += len(encoded)
    return {"status": "complete" if complete else "incomplete", "functions": functions,
            "truncated": not complete}


def _spur_function_identity(value):
    return (value["database_name"], value["schema_name"], value["name"],
            value["function_type"], tuple(value["parameter_types"]), value["varargs"])


def _spur_snapshot_function_registration(db):
    try:
        snapshot = _spur_function_catalog(db)
        if snapshot["status"] == "complete" and snapshot.get("truncated") is False:
            return snapshot
    except Exception:
        pass  # Optional metadata must not prevent an authorized extension load.
    return None


def _spur_record_function_ownership(db, extension, before):
    """Retain registrations observed around a verified load on this connection."""
    if before is None:
        return
    try:
        if before["status"] != "complete" or before.get("truncated") is not False:
            return
        after = _spur_snapshot_function_registration(db)
        if after is None:
            return
        def key(row):
            return (*_spur_function_identity(row), row.get("return_type"), row["function_oid"])
        previous = {key(row) for row in before["functions"]}
        registrations = globals().setdefault("_SPUR_DUCKDB_FUNCTION_OWNERSHIP", {})
        retained = registrations.get(id(db))
        owners = dict(retained[1]) if retained is not None and retained[0] is db else {}
        additions = [key(row) for row in after["functions"]
                     if row["function_oid"] is not None and key(row) not in previous]
        for identity in additions:
            owners[identity] = set(owners.get(identity, ())) | {extension}
        registrations[id(db)] = (db, owners)
    except Exception:
        pass  # Incomplete/failed evidence stays unattributed.


class _SpurDuckDBRuntime:
    BASELINE = ("json", "parquet", "httpfs", "spur_rest")
    REPOSITORIES = {
        # DuckDB's built-in installer handles HTTP without httpfs; HTTPS itself
        # requires httpfs. Use the official signed distribution bootstrap, never
        # temporarily re-enable autoload to download its own dependency.
        "httpfs": "http://extensions.duckdb.org",
        "spatial": "http://extensions.duckdb.org",
        "jev": "http://community-extensions.duckdb.org",
    }

    def __init__(self, db, platform):
        self.db, self.platform = db, platform
        self.entries = {}
        self.inventories = {}  # inert metadata; never activation authority
        self.verified = {}  # at most one private verified copy per curated entry
        self.loaded = set()  # artifact verification is not selected-connection load

    def _entry(self, name):
        return self.entries.setdefault(name, {
            "name": name, "installed": None,
            "desired_enabled": name in self.BASELINE, "loaded": None,
            "provider_health": "unknown", "restart_required": False,
            "activation_transport_restart": False,
            "stage": "idle", "reason": None, "policy": "unknown",
            "artifact_digest": None, "signature_verified": False,
        })

    def _observe(self, db, name):
        return db.sql(
            "SELECT loaded, installed, install_path, installed_from, install_mode "
            "FROM duckdb_extensions() WHERE extension_name = ?", params=[name]
        ).fetchone()

    def _refresh(self, entry):
        row = self._observe(self.db, entry["name"])
        entry["loaded"] = bool(row and row[0])
        entry["installed"] = bool(row and row[1])
        return row

    def _artifact(self, name, install, cancelled):
        # Verification happens in a separate database with signature and ABI
        # checks ON, even when the selected gateway connection allows unsigned
        # bundled code. A repository label or hash alone is not signing evidence.
        import duckdb
        import hashlib
        from pathlib import Path
        import shutil
        import tempfile
        directory = self.db.sql("SELECT current_setting('extension_directory')").fetchone()[0]
        config = {"allow_unsigned_extensions": "false",
                  "allow_community_extensions": "true" if name == "jev" else "false",
                  "allow_extensions_metadata_mismatch": "false",
                  "autoload_known_extensions": "false",
                  "autoinstall_known_extensions": "false"}
        config["home_directory"] = self.db.sql("SELECT current_setting('home_directory')").fetchone()[0]
        if directory:
            config["extension_directory"] = directory
        entry = self._entry(name)
        entry["signature_verified"] = False
        with duckdb.connect(config=config) as verifier:
            if (verifier.sql("SELECT version()").fetchone()[0] != "v1.5.5"
                    or verifier.sql("PRAGMA platform").fetchone()[0] != self.platform):
                raise ValueError("runtime_mismatch")
            entry["stage"] = "installing" if install else "verifying"
            row = self._observe(verifier, name)
            if install and (row is None or not row[1]):
                # Install only when absent. DuckDB rejects INSTALL across HTTP/
                # HTTPS origin spellings even for the same official repository.
                # Cached artifacts follow the same origin/signature/ABI checks
                # below; never FORCE/UPDATE or silently replace rejected bytes.
                verifier.install_extension(name, repository_url=self.REPOSITORIES[name])
                row = self._observe(verifier, name)
            if row is None or not row[1]:
                raise ValueError("not_installed")
            entry["installed"] = True
            if cancelled():
                raise InterruptedError()
            entry["stage"] = "verifying"
            repository = self.REPOSITORIES[name]
            aliases = (repository, repository.replace("http:", "https:"),
                       "community" if name == "jev" else "core")
            if row[3] not in aliases or row[4] != "REPOSITORY":
                raise ValueError("repository_mismatch")
            # Freeze the bytes in a private directory before the verifier loads
            # them. Load those same bytes into the selected connection, never a
            # mutable cache alias. Same-user/kernel arbitrary code is trusted.
            private = tempfile.TemporaryDirectory(prefix="spur-duckdb-verified-")
            try:
                path = Path(private.name) / (name + ".duckdb_extension")
                shutil.copyfile(row[2], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).hexdigest()
                try:
                    before = _spur_function_catalog(verifier)
                except Exception:
                    before = None
                verifier.load_extension(str(path))  # engine checks signature/ABI
                if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
                    raise ValueError("artifact_changed")
                try:
                    after = _spur_function_catalog(verifier)
                    if before is None:
                        raise ValueError("inventory_failed")
                    previous_rows = {_spur_function_identity(f): f for f in before["functions"]}
                    functions = [f for f in after["functions"]
                                 if previous_rows.get(_spur_function_identity(f)) != f]
                    complete = before["status"] == after["status"] == "complete"
                    inventory = {"status": "complete" if complete else "incomplete",
                                 "functions": functions, "truncated": not complete}
                except Exception:
                    inventory = {"status": "failed", "functions": [], "truncated": False}
                inventory.update(artifact_digest=digest, provenance="isolated_signed_verifier_load_delta")
                self.inventories[name] = inventory
            except BaseException:
                private.cleanup()
                raise
            previous = self.verified.get(name)
            if previous is not None:
                previous.cleanup()
            self.verified[name] = private
            return str(path), digest

    def operate(self, name, action, cancelled=lambda: False):
        """Explicit, nonsecret job; called only on the kernel shell queue.

        Cancellation is cooperative at stages. It never interrupts a user query;
        installed but not loaded artifacts remain available for an explicit retry.
        There is no UNLOAD in DuckDB: disable reports loaded + restart_required.
        """
        entry = self._entry(name)
        entry["stage"], entry["reason"] = "resolving", None
        if action == "disable" and name not in self.BASELINE:
            entry["desired_enabled"] = False
            if name == "jev":
                _spur_close_jev_channel()
        try:
            if name not in (*self.BASELINE, "spatial", "jev"):
                raise ValueError("unsupported_extension")
            if self.db is None:
                raise ValueError("no_connection")
            if self.db is not globals().get("_SPUR_DUCKDB_CONNECTION"):
                entry["policy"] = "unenforceable"
                raise ValueError("unmanaged_connection")
            if (self.db.sql("SELECT version()").fetchone()[0] != "v1.5.5"
                    or self.db.sql("PRAGMA platform").fetchone()[0] != self.platform):
                raise ValueError("runtime_mismatch")
            if _spur_managed_connection(self.db) is None:
                entry["policy"] = "unenforceable"
                raise ValueError("policy_unenforceable")
            entry["policy"] = "enforced"
            self._refresh(entry)
            if action == "disable":
                if name in self.BASELINE:
                    raise ValueError("required_baseline")
                entry["desired_enabled"] = False
                entry["restart_required"] = entry["loaded"] or name == "jev"
                entry["activation_transport_restart"] = name == "jev"
                entry["reason"] = "restart_required" if entry["restart_required"] else None
                entry["stage"] = "succeeded"
                return dict(entry)
            if cancelled():
                raise InterruptedError()
            if action == "invalidate_credential_channel" and name == "jev":
                _spur_close_jev_channel()
                entry["restart_required"] = True
                entry["activation_transport_restart"] = True
                entry["reason"] = "restart_required"
            elif action == "enable":
                entry["desired_enabled"] = True
                if name == "jev" and not self._channel_ready():
                    entry["restart_required"] = True
                    entry["activation_transport_restart"] = True
                    entry["reason"] = self._channel_reason()
            elif action in ("install", "revalidate", "load", "prepare_verification"):
                if action == "prepare_verification" and (name not in self.REPOSITORIES or not entry["installed"]
                        or not entry["signature_verified"] or not entry["artifact_digest"]):
                    raise ValueError("not_installed")
                if action == "load" and not entry["desired_enabled"]:
                    raise ValueError("setup_required_without_autoload")
                if name in ("json", "parquet"):
                    row = self._observe(self.db, name)
                    if not row or row[4] != "STATICALLY_LINKED" or not row[0]:
                        raise ValueError("baseline_unavailable")
                elif name == "spur_rest":
                    # Owned by the bundled artifact installer/bootstrap. Never
                    # weaken signature policy or download a substitute here.
                    if not entry["loaded"]:
                        raise ValueError("baseline_unavailable")
                else:
                    if entry["loaded"] and name == "httpfs" and name not in self.loaded:
                        # Connection setup may have loaded the same signed baseline
                        # first. Accept only its exact-connection frozen-byte proof.
                        from pathlib import Path
                        import hashlib
                        proof = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {}).get((id(self.db), name))
                        if proof and proof[0] is self.db and proof[1] == "core":
                            try:
                                digest = hashlib.sha256((Path(proof[2].name) / "httpfs.duckdb_extension").read_bytes()).digest()
                                if digest == proof[3]:
                                    self.verified[name] = proof[2]
                                    self.loaded.add(name)
                                    entry.update(signature_verified=True, artifact_digest=digest.hex(),
                                                 restart_required=False, activation_transport_restart=False)
                            except OSError:
                                pass
                    if entry["loaded"] and name not in self.loaded:
                        entry["signature_verified"] = False
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = False
                        raise ValueError("preexisting_load_unverified")
                    if entry["loaded"] and action in ("load", "prepare_verification"):
                        # Explicit retry may reacquire discovery in the same isolated
                        # signature verifier; it must not change selected enablement.
                        if action == "prepare_verification" and self.inventories.get(name, {}).get("status") != "complete":
                            loaded_digest = entry["artifact_digest"]
                            _, refreshed_digest = self._artifact(name, False, cancelled)
                            if refreshed_digest != loaded_digest:
                                entry["restart_required"] = True
                                entry["activation_transport_restart"] = False
                                raise ValueError("artifact_changed")
                            entry["signature_verified"] = True
                        if name == "jev" and action == "load" and not self._channel_ready():
                            entry["restart_required"] = True
                            entry["activation_transport_restart"] = True
                            entry["reason"] = self._channel_reason()
                        if entry["restart_required"] and entry["activation_transport_restart"]:
                            entry["reason"] = self._channel_reason()
                        entry["stage"] = "succeeded"
                        return dict(entry)
                    if entry["loaded"]:
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = False
                        raise ValueError("restart_required")
                    path, digest = self._artifact(name, action == "install", cancelled)
                    entry["installed"] = True
                    entry["artifact_digest"] = digest
                    entry["signature_verified"] = True
                    if cancelled():
                        raise InterruptedError()
                    if action in ("load", "prepare_verification") and not entry["loaded"]:
                        entry["stage"] = "loading"
                        before = _spur_snapshot_function_registration(self.db)
                        self.db.load_extension(path)
                        self._refresh(entry)
                        if not entry["loaded"]:
                            raise ValueError("load_failed")
                        self.loaded.add(name)
                        _spur_record_function_ownership(self.db, name, before)
                    if name == "jev" and action != "prepare_verification" and not self._channel_ready():
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = True
                        entry["reason"] = self._channel_reason()
            elif action != "inspect":
                raise ValueError("unsupported_operation")
            if entry["restart_required"] and entry["activation_transport_restart"]:
                entry["reason"] = self._channel_reason()
            entry["stage"] = "succeeded"
        except (KeyboardInterrupt, InterruptedError):
            entry["reason"] = "cancelled"
        except ValueError as error:
            # Only our fixed reasons are exposed; never native exception text.
            allowed = {"runtime_mismatch", "unsupported_extension", "unmanaged_connection",
                       "required_baseline", "setup_required_without_autoload", "not_installed",
                       "repository_mismatch", "artifact_changed", "baseline_unavailable",
                       "preexisting_load_unverified", "load_failed", "unsupported_operation", "no_connection",
                       "policy_unenforceable", "restart_required"}
            entry["reason"] = str(error) if str(error) in allowed else "verification_failed"
        except Exception:
            entry["reason"] = {"installing": "install_failed", "loading": "load_failed",
                               "verifying": "verification_failed"}.get(entry["stage"], "policy_failed")
        return dict(entry)

    @staticmethod
    def _channel_ready():
        import os
        if os.name != "posix":
            return False
        try:
            channel = _spur_jev_channel()
            return channel is not None and channel.available()
        except Exception:
            return False

    @staticmethod
    def _channel_reason():
        import os
        return "restart_required" if os.name == "posix" else "credential_transport_unavailable"

    def close(self):
        """Release verified temporary copies after closing this connection."""
        for directory in self.verified.values():
            directory.cleanup()
        self.verified.clear()


def _spur_duckdb_job(platform, name, action, cancel_path=None):
    """Entry point for typed host jobs; no caller SQL and no credential payload."""
    from pathlib import Path
    global _SPUR_DUCKDB_RUNTIME
    db = globals().get("_SPUR_DUCKDB_CONNECTION")
    current = globals().get("_SPUR_DUCKDB_RUNTIME")
    if current is None or current.db is not db or current.platform != platform:
        current = _SPUR_DUCKDB_RUNTIME = _SpurDuckDBRuntime(db, platform)
    return current.operate(name, action, lambda: bool(cancel_path and Path(cancel_path).exists()))


def _spur_duckdb_baseline():
    """Load approved cached baseline only; missing artifacts need explicit setup.

    Used by both root bootstrap and the library's shared SQL session helper.
    The bundled gateway loader stays with its existing host-owned installer.
    """
    db = globals().get("_SPUR_DUCKDB_CONNECTION")
    if db is None or db.sql("SELECT version()").fetchone()[0] != "v1.5.5":
        return
    platform = db.sql("PRAGMA platform").fetchone()[0]
    current = globals().get("_SPUR_DUCKDB_RUNTIME")
    if current is not None and current.db is not db:
        current = None
    for name in ("json", "parquet", "httpfs"):
        if current is None or not current._entry(name)["loaded"]:
            _spur_duckdb_job(platform, name, "load")


class _SpurJevChannel:
    def __init__(self, descriptor):
        import socket
        self.socket = socket.socket(fileno=int(descriptor))
        self.socket.set_inheritable(False)
        self.lease = self._frame()

    def _read(self, count):
        chunks = bytearray()
        while len(chunks) < count:
            chunk = self.socket.recv(count - len(chunks))
            if not chunk:
                raise RuntimeError("Jev private credential channel closed")
            chunks.extend(chunk)
        return bytes(chunks)

    def _frame(self):
        return self._read(int.from_bytes(self._read(4), "big"))

    def _authorize(self, resolve):
        self.socket.sendall(len(self.lease).to_bytes(4, "big") + self.lease + bytes([resolve]))
        status = self._read(1)
        if status == b"\x02":
            return False
        if status != b"\x01":
            raise RuntimeError("denied")
        return True

    def available(self):
        try:
            return self._authorize(False)
        except Exception:
            self.socket.close()
            raise RuntimeError("Jev credential request denied or unavailable. Verify and enable the current managed local kernel in Settings.") from None

    def request(self):
        try:
            if not self._authorize(True):
                return None
            token = self._frame().decode("ascii")
            if not token or any(not 33 <= ord(c) <= 126 for c in token):
                raise ValueError("invalid credential")
            return token
        except Exception:
            self.socket.close()
            raise RuntimeError("Jev credential request denied or unavailable. Verify and enable the current managed local kernel in Settings.") from None


# Repeated generated SQL preambles preserve the captured channel. Failed
# attachment remains failed until kernel restart; never retry a legacy file.
if "_SPUR_JEV_BROKER" not in globals():
    _SPUR_JEV_BROKER = None
    _SPUR_JEV_BROKER_FAILED = False


def _spur_jev_channel():
    import os
    global _SPUR_JEV_BROKER, _SPUR_JEV_BROKER_FAILED
    if _SPUR_JEV_BROKER_FAILED:
        raise RuntimeError("Jev private credential channel is unavailable. Restart the managed local kernel.")
    descriptor = os.environ.pop("SPUR_JEV_BROKER_FD", None)
    if descriptor is not None:
        try:
            if _SPUR_JEV_BROKER is not None:
                raise RuntimeError("duplicate channel")
            _SPUR_JEV_BROKER = _SpurJevChannel(descriptor)
        except Exception:
            _SPUR_JEV_BROKER_FAILED = True
            raise RuntimeError("Jev private credential channel is unavailable. Restart the managed local kernel.") from None
    return _SPUR_JEV_BROKER


def _spur_close_jev_channel():
    """Disable/revoke/replacement is terminal for this inherited descriptor."""
    import os
    global _SPUR_JEV_BROKER_FAILED
    _SPUR_JEV_BROKER_FAILED = True
    if _SPUR_JEV_BROKER is not None:
        _SPUR_JEV_BROKER.socket.close()
    descriptor = os.environ.pop("SPUR_JEV_BROKER_FD", None)
    if descriptor is not None:
        try:
            os.close(int(descriptor))
        except (ValueError, OSError):
            pass


class _SpurJevSession:
    def __init__(self, db):
        self.db = db
        self.token = None
        self.previous = None

    def __enter__(self):
        import os
        _spur_duckdb_baseline()
        runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
        connection = _spur_managed_connection(self.db)
        if runtime is None or runtime.db is not connection:
            return self
        entry = runtime._entry("jev")
        if not (entry["desired_enabled"] and entry["loaded"] and entry["signature_verified"]
                and "jev" in runtime.loaded):
            return self
        try:
            channel = _spur_jev_channel()
            if channel is None or not channel.available():
                entry["restart_required"] = True
                entry["activation_transport_restart"] = True
                entry["reason"] = "restart_required"
                return self
        except Exception:
            # A dead/revoked channel cannot renew in-place. Do not block ordinary
            # SQL, read a legacy file or install/load anything as a fallback.
            entry["restart_required"] = True
            entry["activation_transport_restart"] = True
            entry["reason"] = "restart_required"
            return self
        endpoint = self.db.sql("SELECT current_setting('jev_api_url')").fetchone()[0]
        if endpoint != "https://api.typesafe.ai/v1/systemone":
            raise RuntimeError("Managed Jev credentials require the TypeSafe HTTPS endpoint. Reset jev_api_url before running this cell.")
        self.db.sql("RESET jev_api_key")
        # Resolve only after installation and exact endpoint validation. The
        # broker rechecks lease, enablement and generation on every request.
        try:
            token = channel.request()
        except Exception:
            entry["restart_required"] = True
            entry["reason"] = "restart_required"
            return self
        if token is None:  # disabled while setup was in flight
            return self
        self.previous = os.environ.get("TYPESAFE_API_KEY")
        self.token = token
        os.environ["TYPESAFE_API_KEY"] = token
        return self

    def __exit__(self, error_type, error, traceback):
        import os
        import traceback as traceback_module

        if self.token is None:
            return False
        token = self.token
        self.token = None
        if self.previous is None:
            os.environ.pop("TYPESAFE_API_KEY", None)
        else:
            os.environ["TYPESAFE_API_KEY"] = self.previous
        self.previous = None
        # A remote API can echo a header in an error. Suppress the original
        # traceback chain and remove the exact managed token from the diagnostic.
        if error is not None and token in "".join(traceback_module.format_exception(error_type, error, traceback)):
            message = str(error).replace(token, "[redacted]")
            token = None
            error = None
            traceback = None
            raise RuntimeError(message) from None
        return False

def _spur_restore_catalog_extensions(connection, requirements):
    """Restore optional membership without making unrelated SQL depend on its success."""
    import sys
    failures = globals().setdefault("_SPUR_CATALOG_EXTENSION_FAILURES", {})
    errors = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        key = (id(connection), name, source)
        if _spur_catalog_extensions_ready(connection, [item]):
            failures.pop(key, None)
            continue
        previous = failures.get(key)
        if previous and previous[0] is connection:
            errors[name] = previous[1]
            continue
        try:
            _spur_prepare_catalog_extensions(connection, [item])
        except Exception as error:
            errors[name] = str(error)
            failures[key] = (connection, str(error))
            print(f"SPUR: optional extension {name} could not be restored: {error}. Retry or remove it in Extensions.", file=sys.stderr)
    return errors


def _spur_catalog_extensions_ready(connection, requirements):
    """Read current connection identity and proofs without installing or loading code."""
    import re
    rows = connection.sql(
        "SELECT extension_name, installed, loaded, install_mode, installed_from, "
        "install_path, aliases FROM duckdb_extensions()"
    ).fetchall()
    inventory = {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}
    proofs = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            return False
        row = inventory.get(name)
        if not row or not row[2]:
            return False
        if row[3] == "STATICALLY_LINKED" and source == "core":
            continue
        proof = proofs.get((id(connection), row[0]))
        if not proof or proof[0] is not connection or proof[1] != source:
            return False
    return True


def _spur_prepare_connection_extensions(statements):
    """Use the same verifier for host-generated connection INSTALL/LOAD pairs."""
    import re
    requirements, loads = {}, set()
    for statement in statements:
        install = re.fullmatch(r"INSTALL ([a-z][a-z0-9_]*)(?: FROM (core|community))?", statement)
        load = re.fullmatch(r"LOAD ([a-z][a-z0-9_]*)", statement)
        if install:
            name, source = install.group(1), install.group(2) or "core"
            if name in requirements and requirements[name] != source:
                raise ValueError("Conflicting managed connection repositories")
            requirements[name] = source
        elif load:
            loads.add(load.group(1))
        else:
            raise ValueError("Invalid managed connection extension statement")
    if loads != set(requirements):
        raise ValueError("Managed connection extension installation and load must match")
    connection = globals().get("_SPUR_DUCKDB_CONNECTION")
    if connection is None:
        connection = duckdb.default_connection() if hasattr(duckdb, "default_connection") else duckdb
    _spur_prepare_catalog_extensions(connection, [
        {"packageName": name, "source": source} for name, source in requirements.items()
    ])


def _spur_prepare_catalog_extensions(connection, requirements):
    """Prepare signed packages on the notebook's shared DuckDB connection."""
    import duckdb as engine
    import hashlib
    from pathlib import Path
    import re
    import shutil
    import tempfile

    checked = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            raise ValueError("Invalid managed DuckDB extension identity")
        if name in checked and checked[name] != source:
            raise ValueError("One package cannot use two repositories in the same runtime")
        checked[name] = source
        # Explicit preparation is an intentional retry; ordinary SQL uses restore's memo.
        globals().get("_SPUR_CATALOG_EXTENSION_FAILURES", {}).pop((id(connection), name, source), None)

    def inventory(db):
        rows = db.sql(
            "SELECT extension_name, installed, loaded, install_mode, installed_from, "
            "install_path, aliases FROM duckdb_extensions()"
        ).fetchall()
        return {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}

    def matches(row, source):
        origin = row[4] or ""
        repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
        return origin == source or origin.rstrip("/") in (
            "http://" + repository, "https://" + repository)

    # Proofs belong to this exact connection, surviving redefinition on SQL replay.
    # Retain private immutable copies for the lifetime of the kernel.
    proofs = globals().setdefault("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for name, source in checked.items():
        row = inventory(connection).get(name)
        if row and row[3] == "STATICALLY_LINKED" and source == "core":
            if not row[2]:
                connection.load_extension(name)
            continue
        proof = proofs.get((id(connection), row[0] if row else name))
        if row and row[2]:
            # The required HTTPFS baseline has an existing signed verifier and
            # frozen copy. Share that exact proof, never a bare loaded flag or
            # repository label. This also supports already-running baseline code.
            if proof is None and row[0] == "httpfs" and source == "core":
                runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
                if runtime is not None and runtime.db is connection and "httpfs" in runtime.loaded:
                    baseline = runtime._entry("httpfs")
                    private = runtime.verified.get("httpfs")
                    if baseline.get("signature_verified") is True and baseline.get("loaded") is True and private is not None:
                        try:
                            digest = hashlib.sha256((Path(private.name) / "httpfs.duckdb_extension").read_bytes()).digest()
                            if digest.hex() == baseline.get("artifact_digest"):
                                proof = (connection, source, private, digest)
                                proofs[(id(connection), row[0])] = proof
                        except OSError:
                            pass
            if proof and proof[0] is connection and proof[1] == source:
                continue
            raise RuntimeError(f"{name} is already loaded without this setup's verified artifact; restart the kernel and retry Install in notebook")

        # The shared connection permits the app's bundled unsigned gateway.
        # Always verify third-party bytes in an isolated engine with signing ON.
        config = {
            "allow_unsigned_extensions": "false",
            "allow_extensions_metadata_mismatch": "false",
            "allow_community_extensions": "true" if source == "community" else "false",
            "autoload_known_extensions": "false",
            "autoinstall_known_extensions": "false",
            "home_directory": connection.sql("SELECT current_setting('home_directory')").fetchone()[0],
        }
        directory = connection.sql("SELECT current_setting('extension_directory')").fetchone()[0]
        if directory:
            config["extension_directory"] = directory
        private = None
        try:
            with engine.connect(config=config) as verifier:
                for query in ("SELECT version()", "PRAGMA platform"):
                    if verifier.sql(query).fetchone() != connection.sql(query).fetchone():
                        raise RuntimeError("DuckDB runtime changed during extension verification")
                artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
                    verifier.install_extension(name, repository_url="http://" + repository)
                    artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    raise RuntimeError(f"{name} installation is unavailable for this DuckDB runtime")
                if artifact[3] != "REPOSITORY" or not matches(artifact, source):
                    raise RuntimeError(f"{name} is cached from another repository; resolve its source before retrying")
                private = tempfile.TemporaryDirectory(prefix="spur-duckdb-catalog-")
                path = Path(private.name) / (artifact[0] + ".duckdb_extension")
                shutil.copyfile(artifact[5], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).digest()
                verifier.load_extension(str(path))  # signature and ABI checked by DuckDB
                if hashlib.sha256(path.read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
            snapshot = globals().get("_spur_snapshot_function_registration")
            before = snapshot(connection) if callable(snapshot) else None
            connection.load_extension(str(path))  # load exactly the verified bytes
            observed = inventory(connection).get(name)
            if not observed or not observed[2]:
                raise RuntimeError(f"{name} did not load into the notebook connection")
            proofs[(id(connection), observed[0])] = (connection, source, private, digest)
            record = globals().get("_spur_record_function_ownership")
            if callable(record):
                record(connection, observed[0], before)
            private = None
        finally:
            if private is not None:
                private.cleanup()


_spur_duckdb_baseline()
def _spur_restore_catalog_extensions(connection, requirements):
    """Restore optional membership without making unrelated SQL depend on its success."""
    import sys
    failures = globals().setdefault("_SPUR_CATALOG_EXTENSION_FAILURES", {})
    errors = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        key = (id(connection), name, source)
        if _spur_catalog_extensions_ready(connection, [item]):
            failures.pop(key, None)
            continue
        previous = failures.get(key)
        if previous and previous[0] is connection:
            errors[name] = previous[1]
            continue
        try:
            _spur_prepare_catalog_extensions(connection, [item])
        except Exception as error:
            errors[name] = str(error)
            failures[key] = (connection, str(error))
            print(f"SPUR: optional extension {name} could not be restored: {error}. Retry or remove it in Extensions.", file=sys.stderr)
    return errors


def _spur_catalog_extensions_ready(connection, requirements):
    """Read current connection identity and proofs without installing or loading code."""
    import re
    rows = connection.sql(
        "SELECT extension_name, installed, loaded, install_mode, installed_from, "
        "install_path, aliases FROM duckdb_extensions()"
    ).fetchall()
    inventory = {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}
    proofs = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            return False
        row = inventory.get(name)
        if not row or not row[2]:
            return False
        if row[3] == "STATICALLY_LINKED" and source == "core":
            continue
        proof = proofs.get((id(connection), row[0]))
        if not proof or proof[0] is not connection or proof[1] != source:
            return False
    return True


def _spur_prepare_connection_extensions(statements):
    """Use the same verifier for host-generated connection INSTALL/LOAD pairs."""
    import re
    requirements, loads = {}, set()
    for statement in statements:
        install = re.fullmatch(r"INSTALL ([a-z][a-z0-9_]*)(?: FROM (core|community))?", statement)
        load = re.fullmatch(r"LOAD ([a-z][a-z0-9_]*)", statement)
        if install:
            name, source = install.group(1), install.group(2) or "core"
            if name in requirements and requirements[name] != source:
                raise ValueError("Conflicting managed connection repositories")
            requirements[name] = source
        elif load:
            loads.add(load.group(1))
        else:
            raise ValueError("Invalid managed connection extension statement")
    if loads != set(requirements):
        raise ValueError("Managed connection extension installation and load must match")
    connection = globals().get("_SPUR_DUCKDB_CONNECTION")
    if connection is None:
        connection = duckdb.default_connection() if hasattr(duckdb, "default_connection") else duckdb
    _spur_prepare_catalog_extensions(connection, [
        {"packageName": name, "source": source} for name, source in requirements.items()
    ])


def _spur_prepare_catalog_extensions(connection, requirements):
    """Prepare signed packages on the notebook's shared DuckDB connection."""
    import duckdb as engine
    import hashlib
    from pathlib import Path
    import re
    import shutil
    import tempfile

    checked = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            raise ValueError("Invalid managed DuckDB extension identity")
        if name in checked and checked[name] != source:
            raise ValueError("One package cannot use two repositories in the same runtime")
        checked[name] = source
        # Explicit preparation is an intentional retry; ordinary SQL uses restore's memo.
        globals().get("_SPUR_CATALOG_EXTENSION_FAILURES", {}).pop((id(connection), name, source), None)

    def inventory(db):
        rows = db.sql(
            "SELECT extension_name, installed, loaded, install_mode, installed_from, "
            "install_path, aliases FROM duckdb_extensions()"
        ).fetchall()
        return {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}

    def matches(row, source):
        origin = row[4] or ""
        repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
        return origin == source or origin.rstrip("/") in (
            "http://" + repository, "https://" + repository)

    # Proofs belong to this exact connection, surviving redefinition on SQL replay.
    # Retain private immutable copies for the lifetime of the kernel.
    proofs = globals().setdefault("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for name, source in checked.items():
        row = inventory(connection).get(name)
        if row and row[3] == "STATICALLY_LINKED" and source == "core":
            if not row[2]:
                connection.load_extension(name)
            continue
        proof = proofs.get((id(connection), row[0] if row else name))
        if row and row[2]:
            # The required HTTPFS baseline has an existing signed verifier and
            # frozen copy. Share that exact proof, never a bare loaded flag or
            # repository label. This also supports already-running baseline code.
            if proof is None and row[0] == "httpfs" and source == "core":
                runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
                if runtime is not None and runtime.db is connection and "httpfs" in runtime.loaded:
                    baseline = runtime._entry("httpfs")
                    private = runtime.verified.get("httpfs")
                    if baseline.get("signature_verified") is True and baseline.get("loaded") is True and private is not None:
                        try:
                            digest = hashlib.sha256((Path(private.name) / "httpfs.duckdb_extension").read_bytes()).digest()
                            if digest.hex() == baseline.get("artifact_digest"):
                                proof = (connection, source, private, digest)
                                proofs[(id(connection), row[0])] = proof
                        except OSError:
                            pass
            if proof and proof[0] is connection and proof[1] == source:
                continue
            raise RuntimeError(f"{name} is already loaded without this setup's verified artifact; restart the kernel and retry Install in notebook")

        # The shared connection permits the app's bundled unsigned gateway.
        # Always verify third-party bytes in an isolated engine with signing ON.
        config = {
            "allow_unsigned_extensions": "false",
            "allow_extensions_metadata_mismatch": "false",
            "allow_community_extensions": "true" if source == "community" else "false",
            "autoload_known_extensions": "false",
            "autoinstall_known_extensions": "false",
            "home_directory": connection.sql("SELECT current_setting('home_directory')").fetchone()[0],
        }
        directory = connection.sql("SELECT current_setting('extension_directory')").fetchone()[0]
        if directory:
            config["extension_directory"] = directory
        private = None
        try:
            with engine.connect(config=config) as verifier:
                for query in ("SELECT version()", "PRAGMA platform"):
                    if verifier.sql(query).fetchone() != connection.sql(query).fetchone():
                        raise RuntimeError("DuckDB runtime changed during extension verification")
                artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
                    verifier.install_extension(name, repository_url="http://" + repository)
                    artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    raise RuntimeError(f"{name} installation is unavailable for this DuckDB runtime")
                if artifact[3] != "REPOSITORY" or not matches(artifact, source):
                    raise RuntimeError(f"{name} is cached from another repository; resolve its source before retrying")
                private = tempfile.TemporaryDirectory(prefix="spur-duckdb-catalog-")
                path = Path(private.name) / (artifact[0] + ".duckdb_extension")
                shutil.copyfile(artifact[5], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).digest()
                verifier.load_extension(str(path))  # signature and ABI checked by DuckDB
                if hashlib.sha256(path.read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
            snapshot = globals().get("_spur_snapshot_function_registration")
            before = snapshot(connection) if callable(snapshot) else None
            connection.load_extension(str(path))  # load exactly the verified bytes
            observed = inventory(connection).get(name)
            if not observed or not observed[2]:
                raise RuntimeError(f"{name} did not load into the notebook connection")
            proofs[(id(connection), observed[0])] = (connection, source, private, digest)
            record = globals().get("_spur_record_function_ownership")
            if callable(record):
                record(connection, observed[0], before)
            private = None
        finally:
            if private is not None:
                private.cleanup()

import json as _spur_extension_json
_spur_prepare_catalog_extensions(_SPUR_DUCKDB_CONNECTION, _spur_extension_json.loads("[{\"packageName\":\"duckrouting\",\"source\":\"community\"},{\"packageName\":\"shellfs\",\"source\":\"community\"}]"))


# Panic-path exploration: Spur graph + shellfs

This notebook explores [Finding panics with Call Graph Analyzer](https://tweedegolf.nl/en/blog/242/finding-panics-with-call-graph-analyzer/) using the Spur source graph. **This is a source-level experiment, not CGA's MIR analysis or a panic-freedom proof.**

Run the cells from top to bottom. The managed setup cell restores the signed `shellfs` and `duckrouting` extensions. The remaining cells require `rg` in the notebook kernel's PATH and the local repository/analyst paths shown in the SQL. Source scans are live; the graph database is attached read-only.

- `shellfs` streams a fixed, read-only `rg --json` command into DuckDB.
- The analyst index supplies candidate call edges and symbol ranges. The index used for this run reported indexed commit `46b85c5` while the working checkout was `ea25f4f`. Refresh the graph before drawing current coverage conclusions; restart the kernel or detach the existing read-only attachment to pick up a refreshed database.
- `duckrouting` computes directed paths over recorded source-level call edges, including heuristic dynamic edges. No recorded path means only that this graph did not find one.
- Text matches include tests and comments; unresolved or heuristic edges are not compiler-verified. The candidate query includes only `.expect` and `.unwrap` labels; it does not cover implicit checks, dependency internals, or all panic syntax.


In [4]:
SELECT extension_name, installed, loaded, install_mode FROM duckdb_extensions() WHERE extension_name IN ('shellfs', 'duckrouting') ORDER BY extension_name;

extension_name,installed,loaded,install_mode
duckrouting,True,True,REPOSITORY
shellfs,True,True,REPOSITORY


In [5]:
SELECT * FROM read_csv($$printf 'name,score\nprobe,7\n' |$$, header = true);

name,score
probe,7


In [7]:
ATTACH IF NOT EXISTS '/Users/kevintruong/project/SpurCode/.spur/analyst.duckdb' AS spur_graph (READ_ONLY); SELECT graph_content_hash, graph_index_version, extractor_version, complete FROM spur_graph._meta;

graph_content_hash,graph_index_version,extractor_version,complete
8599245794571718b3153894d70d281c091e04ef3300eff4d66c0f3eb9fe6d59,4,2026-08-30-html-css-extraction-v1,True


In [8]:
SELECT * FROM read_csv($$command -v rg || printf 'RG_NOT_IN_PATH\n' |$$, columns = {'path': 'VARCHAR'}, header = false);

path
/opt/homebrew/bin/rg


In [9]:
SELECT table_name, column_name, data_type FROM information_schema.columns WHERE table_catalog = 'spur_graph' AND table_name IN ('nodes','edges','edges_unresolved') ORDER BY table_name, ordinal_position;

table_name,column_name,data_type
edges,source_stable_id,VARCHAR
edges,target_stable_id,VARCHAR
edges,src_id,BIGINT
edges,dst_id,BIGINT
edges,target_label,VARCHAR
edges,relation,VARCHAR
edges,confidence,VARCHAR
edges,confidence_score,FLOAT
edges,edge_kind,VARCHAR
edges,bind_method,VARCHAR


In [10]:
SELECT node_id, stable_symbol_id, qualified_name, file_path, line_start FROM spur_graph.nodes WHERE stable_symbol_id IN ('63f6898cbb3fcfdd', '5fb06d53a2477ff7', '10dad659a68a8b0f') ORDER BY file_path, line_start;

node_id,stable_symbol_id,qualified_name,file_path,line_start
36667,63f6898cbb3fcfdd,build_with_temporal_jobs_override,crates/spur-cli/src/commands/graph.rs,108
35096,5fb06d53a2477ff7,run_full_walk_into,crates/spur-graph/src/git_walk.rs,910
6133,10dad659a68a8b0f,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,920


In [11]:
CREATE OR REPLACE TEMP TABLE panic_source_hits AS
SELECT
  replace(json_extract_string(data, '$.path.text'), '/Users/kevintruong/project/SpurCode/', '') AS file_path,
  try_cast(json_extract_string(data, '$.line_number') AS INTEGER) AS line_number,
  trim(json_extract_string(data, '$.lines.text')) AS source_line
FROM read_json(
  $$rg --json --glob '*.rs' -e '\.(unwrap|expect)\s*\(' -e '\b(panic|unreachable|todo|unimplemented)!\s*[\(\{\[]' /Users/kevintruong/project/SpurCode/crates/spur-graph/src |$$,
  format = 'newline_delimited',
  columns = {'type': 'VARCHAR', 'data': 'JSON'}
)
WHERE type = 'match';

SELECT count(*) AS matching_lines_including_tests_and_comments,
       count(DISTINCT file_path) AS files
FROM panic_source_hits;

matching_lines_including_tests_and_comments,files
1818,39


In [13]:
-- Node IDs are derived from the current graph snapshot; they can change on rebuild.
SET VARIABLE route_start = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = '63f6898cbb3fcfdd');
SET VARIABLE route_target = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = '10dad659a68a8b0f');

SELECT path.seq, path.node, n.qualified_name, n.file_path, path.agg_cost AS hops
FROM duckrouting_dijkstra(
  $$SELECT row_number() OVER ()::BIGINT AS id,
           src_id AS source, dst_id AS target,
           1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
    FROM spur_graph.edges
    WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')$$,
  getvariable('route_start'), getvariable('route_target')
) AS path
JOIN spur_graph.nodes n ON n.node_id = path.node
ORDER BY path.seq;

seq,node,qualified_name,file_path,hops
1,36667,build_with_temporal_jobs_override,crates/spur-cli/src/commands/graph.rs,0.0
2,35096,run_full_walk_into,crates/spur-graph/src/git_walk.rs,1.0
3,6133,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,2.0


In [14]:
SELECT DISTINCT n.qualified_name, h.file_path, h.line_number, e.target_label,
       e.confidence AS graph_confidence, h.source_line
FROM panic_source_hits h
JOIN spur_graph.nodes n ON n.file_path = h.file_path
  AND h.line_number BETWEEN n.line_start AND n.line_end
JOIN spur_graph.edges_unresolved e ON e.source_stable_id = n.stable_symbol_id
WHERE n.stable_symbol_id = '10dad659a68a8b0f'
  AND e.target_label IN ('unwrap', 'expect')
  AND regexp_matches(h.source_line, '\.' || e.target_label || '\s*\(')
ORDER BY h.line_number;

qualified_name,file_path,line_number,target_label,graph_confidence,source_line
run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,958,expect,syntax_exact,"let base = base.expect(""incremental base checked above"");\n"


In [16]:
-- These are candidates, including possible test/comment matches, not proven panics.
SET VARIABLE route_start = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = '63f6898cbb3fcfdd');

WITH reachable AS (
  SELECT node, agg_cost AS hops
  FROM duckrouting_driving_distance(
    $$SELECT row_number() OVER ()::BIGINT AS id,
             src_id AS source, dst_id AS target,
             1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
      FROM spur_graph.edges
      WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')$$,
    getvariable('route_start'), 3.0
  )
), candidates AS (
  SELECT r.hops, n.qualified_name, n.file_path, h.line_number, e.target_label,
         h.source_line,
         row_number() OVER (
           PARTITION BY h.file_path, h.line_number, e.target_label
           ORDER BY n.line_end - n.line_start
         ) AS smallest_symbol
  FROM reachable r
  JOIN spur_graph.nodes n ON n.node_id = r.node
  JOIN spur_graph.edges_unresolved e ON e.source_stable_id = n.stable_symbol_id
  JOIN panic_source_hits h ON h.file_path = n.file_path
       AND h.line_number BETWEEN n.line_start AND n.line_end
       AND regexp_matches(h.source_line, '\.' || e.target_label || '\s*\(')
  WHERE n.file_path LIKE 'crates/spur-graph/src/%'
    AND e.target_label IN ('unwrap', 'expect')
)
SELECT hops, qualified_name, file_path, line_number, target_label, source_line
FROM candidates
WHERE smallest_symbol = 1
ORDER BY hops, file_path, line_number
LIMIT 30;

hops,qualified_name,file_path,line_number,target_label,source_line
2.0,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,958,expect,"let base = base.expect(""incremental base checked above"");\n"
2.0,write_edges,crates/spur-graph/src/store/parquet.rs,1720,expect,".expect(""resolved edge has target"")\n"
3.0,rebuild_from_buckets,crates/spur-graph/src/store/build.rs,2530,expect,".expect(""exhausted synthetic file NodeId space"");\n"
3.0,stable_file_id_from_path,crates/spur-graph/src/store/build.rs,2671,unwrap,u64::from_be_bytes(hasher.finalize()[..8].try_into().unwrap())\n


In [17]:
SET VARIABLE route_start = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = '63f6898cbb3fcfdd');
SET VARIABLE route_target = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = 'f7e5a680e7069785');
SELECT path.seq, n.qualified_name, n.file_path, path.agg_cost AS hops
FROM duckrouting_dijkstra(
  $$SELECT row_number() OVER ()::BIGINT AS id,
           src_id AS source, dst_id AS target,
           1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
    FROM spur_graph.edges
    WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')$$,
  getvariable('route_start'), getvariable('route_target')
) AS path
JOIN spur_graph.nodes n ON n.node_id = path.node
ORDER BY path.seq;

seq,qualified_name,file_path,hops
1,build_with_temporal_jobs_override,crates/spur-cli/src/commands/graph.rs,0.0
2,artifact_from_facts_incremental,crates/spur-graph/src/store/build.rs,1.0
3,compose_artifact,crates/spur-graph/src/store/build.rs,2.0
4,rebuild_from_buckets,crates/spur-graph/src/store/build.rs,3.0


In [18]:
-- The solver's witness, evaluated with wide SQL integers.
-- It says nothing about whether such an input can actually enter this function.
WITH model AS (
  SELECT CAST('18446744073709551614' AS HUGEINT) AS max_existing,
         true AS missing_file_id
)
SELECT max_existing, missing_file_id, max_existing + 1 AS next_synthetic,
       try_cast(max_existing + 2 AS UBIGINT) IS NULL AS next_checked_add_would_fail
FROM model;

max_existing,missing_file_id,next_synthetic,next_checked_add_would_fail
18446744073709551614,True,18446744073709551615,True


In [19]:
-- Exhaust the Boolean abstraction used by the previous solver check.
WITH inputs(fast_forward, base_present) AS (
  VALUES (false, false), (false, true), (true, false), (true, true)
), model AS (
  SELECT *, fast_forward AND base_present AS at_expect FROM inputs
)
SELECT count(*) AS assignments,
       count(*) FILTER (WHERE at_expect) AS assignments_reaching_expect,
       count(*) FILTER (WHERE at_expect AND NOT base_present) AS assignments_that_panic
FROM model;

assignments,assignments_reaching_expect,assignments_that_panic
4,1,0


## Interpretation and solver checks

The three-hop candidate query above identified four matching lines on the indexed call graph in this run. Those are **candidates**, not four demonstrated panics. The graph has unresolved call labels and source spans belong to symbols, so matching a current line to an older indexed function is a heuristic.

- `git_walk.rs:958`: at the `expect`, `use_incremental_base = fast_forward AND base_present`. A preflighted `solve_constraints` query found the branch satisfiable, and the branch plus `NOT base_present` **unsatisfiable**. The Boolean enumeration below independently evaluates all four input combinations. This result covers that manual abstraction, not other calls or paths.
- `store/build.rs:2530`: the 64-bit solver query returned **sat** for `max_existing = 0xfffffffffffffffe` and a file with no node ID. The next synthetic ID is `0xffffffffffffffff`, so incrementing it fails. The SQL witness below evaluates this arithmetic. This is a model-level boundary case; the experiment has not shown that a normal or accepted artifact can produce it.
- `store/parquet.rs:1720`: `edge_rows` creates resolved rows from `Some(target)`; proving every possible path into `write_edges` preserves that invariant needs a separate review.
- `store/build.rs:2671`: the source slices eight bytes from a SHA-256 digest before conversion. This suggests an invariant-based triage, but it was not separately solver-verified here.

Solver model for the 64-bit case (all constraints hard): `max_existing != all_ones`, `next_synthetic == bv_add(max_existing, 1)`, `missing_file_id == true`, and `next_synthetic == all_ones`. The typed request passed `solve_constraint_check` before `solve_constraints` returned the model. The solver operated separately through MCP; the SQL cell evaluates its recorded model rather than invoking Z3.

Useful next steps are a fresh graph rebuild, compiler-backed CGA output for the chosen binary, and a targeted test or invariant check for the boundary case. CGA and the source graph have different completeness and false-positive limits.
